# Notebook 04 – Feature Importance (Random Forest + SHAP)

Identify which user context variables most influence UI preferences using grouped predictive modelling:

- **Global UI**
- **Desktop UI**
- **Mobile UI**

Each group is processed automatically in a single pipeline loop.


## Expected Outputs
- Consolidated: `feature_importance`, `shap_summary`, `model_metrics`, `target_predictor_summary`
- Group reports: `group_reports/global_ui_summary`, `desktop_ui_summary`, `mobile_ui_summary`
- Plots: `plots/Global_UI/`, `plots/Desktop_UI/`, `plots/Mobile_UI/`
- `pipeline_summary.md`


In [1]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


_bootstrap_project()

from src.utils.dependencies import ensure_packages

ensure_packages("shap", "scikit-learn")

from src.machine_learning.pipeline import run_feature_importance_pipeline
from src.preprocessing.columns import get_ml_feature_columns, get_ui_target_groups
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)
plt.rcParams["figure.dpi"] = 300

PATHS, REPORTS = setup_notebook("Feature_Importance")
feature_columns = get_ml_feature_columns()
ui_target_groups = get_ui_target_groups()

for group_name, targets in ui_target_groups.items():
    print(f"{group_name}: {len(targets)} targets")



Global_UI: 14 targets
Desktop_UI: 13 targets
Mobile_UI: 14 targets


/Users/mariam/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data


In [2]:
df = pd.read_csv(PATHS.data_processed / "clean_dataset.csv")
statistical_results = pd.read_csv(
    PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.csv"
)
logger.info("Loaded clean dataset: %s rows", len(df))
display(df[feature_columns].head())


INFO: Loaded clean dataset: 224 rows


,primary_persona,current_mood,primary_device,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness
0,The Impulsive Buyer (I make quick decisions ba...,Neutral,Smartphone,3.0,3.0,3.0,3.0,3.0
1,The Loyal Customer (I stick with brands and st...,Happy,Smartphone,3.5,4.5,3.5,2.5,2.5
2,The Browser (I enjoy browsing and discovering ...,Stressed,Smartphone,4.0,3.0,4.5,5.0,4.0
3,"The Minimalist (I want simple, efficient shopp...",Happy,Smartphone,4.0,3.0,3.0,3.5,3.0
4,The Researcher (I thoroughly research products...,Bored,Smartphone,2.5,5.0,4.0,2.0,2.5


## Run Grouped Feature Importance Pipeline


In [3]:
pipeline_result = run_feature_importance_pipeline(
    df=df,
    feature_columns=feature_columns,
    reports_dir=REPORTS,
    ui_target_groups=ui_target_groups,
)

model_metrics = pipeline_result.model_metrics
feature_importance = pipeline_result.feature_importance
shap_summary = pipeline_result.shap_summary
target_summary = pipeline_result.target_summary
overall_importance = pipeline_result.overall_importance
group_importance = pipeline_result.group_importance

print(f"Models trained: {len(model_metrics)}")
display(model_metrics.groupby('UI_Group')['accuracy'].mean())


INFO: Processing UI group: Global_UI (14 targets)
INFO: Trained font_style_pref | accuracy=0.422 | best={'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
INFO: Trained font_size_pref | accuracy=0.400 | best={'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
INFO: Trained color_theme_pref | accuracy=0.200 | best={'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
INFO: Trained accent_color_pref | accuracy=0.622 | best={'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
INFO: Trained background_pref | accuracy=0.356 | best={'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
INFO: Trained whitespace_pref | accuracy=0.756 | best={'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_spl

Models trained: 41


UI_Group
Desktop_UI    0.482051
Global_UI     0.390476
Mobile_UI     0.536508
Name: accuracy, dtype: float64

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

...

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

## Consolidated Results Preview


In [4]:
display(feature_importance.groupby('UI_Group').head(3))
display(target_summary.head(10))
display(group_importance)


,UI_Group,UI_Target,Predictor,Importance,Rank
0,Global_UI,font_style_pref,Neuroticism,0.154303,1
1,Global_UI,font_style_pref,Openness,0.152852,2
2,Global_UI,font_style_pref,Extraversion,0.140394,3
112,Desktop_UI,desktop_grid_pref,Conscientiousness,0.151289,1
113,Desktop_UI,desktop_grid_pref,Neuroticism,0.145067,2
114,Desktop_UI,desktop_grid_pref,Agreeableness,0.140391,3
216,Mobile_UI,mobile_grid_pref,Neuroticism,0.165426,1
217,Mobile_UI,mobile_grid_pref,Agreeableness,0.141139,2
218,Mobile_UI,mobile_grid_pref,Conscientiousness,0.138440,3


,UI_Group,UI_Target,Top_Predictor,Top_Predictor_Importance,Top_3_Predictors,Top_3_Importance_Values,Top_SHAP_Predictors,Top_SHAP_Values,Top_Mean_ABS_SHAP
0,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099
1,Global_UI,font_size_pref,Neuroticism,0.156377,"Neuroticism, Agreeableness, Extraversion","0.1564, 0.1440, 0.1414","current_mood, Conscientiousness, Neuroticism","0.0276, 0.0261, 0.0249",0.027552
2,Global_UI,color_theme_pref,Neuroticism,0.147515,"Neuroticism, primary_persona, Conscientiousness","0.1475, 0.1452, 0.1419","Neuroticism, Openness, Extraversion","0.0301, 0.0238, 0.0220",0.030115
3,Global_UI,accent_color_pref,Agreeableness,0.145652,"Agreeableness, Neuroticism, current_mood","0.1457, 0.1423, 0.1419","Neuroticism, primary_persona, Agreeableness","0.0205, 0.0205, 0.0198",0.020525
4,Global_UI,background_pref,Neuroticism,0.164472,"Neuroticism, primary_persona, Conscientiousness","0.1645, 0.1426, 0.1349","Agreeableness, Neuroticism, Extraversion","0.0356, 0.0295, 0.0256",0.035629
5,Global_UI,whitespace_pref,Conscientiousness,0.158236,"Conscientiousness, Neuroticism, Openness","0.1582, 0.1499, 0.1376","current_mood, Neuroticism, Extraversion","0.0382, 0.0323, 0.0305",0.038245
6,Global_UI,button_style_pref,Neuroticism,0.151966,"Neuroticism, Openness, Conscientiousness","0.1520, 0.1427, 0.1397","Agreeableness, Extraversion, Neuroticism","0.0229, 0.0229, 0.0220",0.022936
7,Global_UI,hero_banner_size,Neuroticism,0.165352,"Neuroticism, Openness, Extraversion","0.1654, 0.1465, 0.1443","primary_device, Openness, Neuroticism","0.0344, 0.0311, 0.0282",0.034364
8,Global_UI,recommendation_type,Neuroticism,0.155906,"Neuroticism, Conscientiousness, primary_persona","0.1559, 0.1455, 0.1407","Neuroticism, current_mood, Agreeableness","0.0443, 0.0345, 0.0309",0.044346
9,Global_UI,social_proof_display,Neuroticism,0.151059,"Neuroticism, primary_persona, Conscientiousness","0.1511, 0.1466, 0.1448","Neuroticism, primary_persona, Agreeableness","0.0376, 0.0305, 0.0258",0.037582


,UI_Group,Predictor,Importance
3,Desktop_UI,Neuroticism,0.153506
0,Desktop_UI,Agreeableness,0.138037
1,Desktop_UI,Conscientiousness,0.136338
4,Desktop_UI,Openness,0.136103
5,Desktop_UI,current_mood,0.134527
2,Desktop_UI,Extraversion,0.133689
7,Desktop_UI,primary_persona,0.130738
6,Desktop_UI,primary_device,0.037062
11,Global_UI,Neuroticism,0.152761
9,Global_UI,Conscientiousness,0.140502


## Compare with Notebook 03 (Reference Only)


In [5]:
comparison = statistical_results[[
    'Predictor', 'UI_Element', 'Raw_P', 'Cramers_V', 'Evidence_Score'
]].rename(columns={'UI_Element': 'UI_Target', 'Predictor': 'Statistical_Predictor'})

comparison_preview = target_summary.merge(
    comparison,
    on='UI_Target',
    how='left',
)
display(comparison_preview.head(10))


,UI_Group,UI_Target,Top_Predictor,Top_Predictor_Importance,Top_3_Predictors,Top_3_Importance_Values,Top_SHAP_Predictors,Top_SHAP_Values,Top_Mean_ABS_SHAP,Statistical_Predictor,Raw_P,Cramers_V,Evidence_Score
0,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,primary_persona,0.085737,0.184703,0.710680
1,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,current_mood,0.486055,0.174917,0.480038
2,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,Openness_Level,0.101383,0.153856,0.713946
3,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,primary_device,0.204601,0.143111,0.510211
4,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,Conscientiousness_Level,0.544690,0.105572,0.388007
5,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,Neuroticism_Level,0.574677,0.103100,0.344954
6,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,Agreeableness_Level,0.785536,0.084292,0.221832
7,Global_UI,font_style_pref,Neuroticism,0.154303,"Neuroticism, Openness, Extraversion","0.1543, 0.1529, 0.1404","primary_device, Neuroticism, primary_persona","0.0381, 0.0299, 0.0282",0.038099,Extraversion_Level,0.994602,0.039382,0.058152
8,Global_UI,font_size_pref,Neuroticism,0.156377,"Neuroticism, Agreeableness, Extraversion","0.1564, 0.1440, 0.1414","current_mood, Conscientiousness, Neuroticism","0.0276, 0.0261, 0.0249",0.027552,current_mood,0.820096,0.149686,0.308666
9,Global_UI,font_size_pref,Neuroticism,0.156377,"Neuroticism, Agreeableness, Extraversion","0.1564, 0.1440, 0.1414","current_mood, Conscientiousness, Neuroticism","0.0276, 0.0261, 0.0249",0.027552,Conscientiousness_Level,0.139700,0.146845,0.643051


## Final Summary


In [6]:
best_model = model_metrics.loc[model_metrics['accuracy'].idxmax()]
avg_accuracy = model_metrics['accuracy'].mean()

print(f"Best performing UI prediction: {best_model['UI_Target']} ({best_model['UI_Group']}, accuracy={best_model['accuracy']:.3f})")
print(f"Average model accuracy: {avg_accuracy:.3f}")
print("Top 20 most influential predictors overall:")
print(overall_importance.to_string(index=False))
print("\nExport locations:")
for name, path in pipeline_result.export_paths.items():
    print(f"- {name}: {path}")


Best performing UI prediction: mobile_image_text_ratio (Mobile_UI, accuracy=0.844)
Average model accuracy: 0.469
Top 20 most influential predictors overall:
        Predictor  Importance
      Neuroticism    0.155522
Conscientiousness    0.140467
         Openness    0.137032
    Agreeableness    0.136675
  primary_persona    0.133338
     Extraversion    0.131607
     current_mood    0.130229
   primary_device    0.035131

Export locations:
- csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Feature_Importance/group_importance.csv
- xlsx: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Feature_Importance/group_importance.xlsx
- pipeline_summary_md: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Feature_Importance/pipeline_summary.md
